# One row, then one person

The first line of `u.data` is not user 1 and not the earliest rating. The file is stored in a random order. That line is

$$
\text{user } 196, \quad \text{movie } 242, \quad \text{rating } 3, \quad \text{timestamp } 881250949.
$$

Movie 242 is *Kolya (1996)*, released 24 January 1997. The timestamp is Unix seconds. In UTC it is 4 December 1997, 15:55:49. User 196 is 49, male, a writer, zip code 55105.

User 1 is a different person: 24, male, a technician, zip code 85711. Across the whole rating file this user has 272 ratings, summing to 982.

$$
\dfrac{982}{272} = \dfrac{491}{136}.
$$

The first five movies in id order, among the movies user 1 rated, are

| Movie | Title | Stars |
|---:|---|---:|
| 1 | Toy Story (1995) | 5 |
| 2 | GoldenEye (1995) | 3 |
| 3 | Four Rooms (1995) | 4 |
| 4 | Get Shorty (1995) | 3 |
| 5 | Copycat (1995) | 3 |

Toy Story's genre flags are Animation, Children's, and Comedy. GoldenEye's are Action, Adventure, and Thriller. Those flags are a description of the movie. They are not the rating.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# The first stored rating, user 196, and user 1's mean.
from fractions import Fraction
from datetime import datetime, timezone
movies = {int(row[0]): row for row in (line.split("|") for line in read_lines("u.item"))}
users = {int(row[0]): row for row in (line.split("|") for line in read_lines("u.user"))}
genres = [line.split("|")[0] for line in read_lines("u.genre") if line.strip()]
first = ratings[0]
print(first, movies[242][1], users[196])
stamp = datetime.fromtimestamp(881250949, timezone.utc)
print(stamp.isoformat())
assert first == (196, 242, 3, 881250949)
assert movies[242][1] == "Kolya (1996)"
assert users[196] == ["196", "49", "M", "writer", "55105"]
assert stamp.isoformat() == "1997-12-04T15:55:49+00:00"
user_one = [rating for user, _, rating, _ in ratings if user == 1]
print(len(user_one), sum(user_one), Fraction(sum(user_one), len(user_one)))
assert len(user_one) == 272
assert sum(user_one) == 982
assert Fraction(982, 272) == Fraction(491, 136)
by_item = {item: rating for user, item, rating, _ in ratings if user == 1}
for item, expected in ((1, 5), (2, 3), (3, 4), (4, 3), (5, 3)):
    print(item, movies[item][1], by_item[item])
    assert by_item[item] == expected
toy_flags = [int(flag) for flag in movies[1][5:]]
gold_flags = [int(flag) for flag in movies[2][5:]]
toy_names = [name for name, flag in zip(genres, toy_flags) if flag]
gold_names = [name for name, flag in zip(genres, gold_flags) if flag]
print(toy_names, gold_names)
assert toy_names == ["Animation", "Children's", "Comedy"]
assert gold_names == ["Action", "Adventure", "Thriller"]
assert users[1] == ["1", "24", "M", "technician", "85711"]


(196, 242, 3, 881250949) Kolya (1996) ['196', '49', 'M', 'writer', '55105']
1997-12-04T15:55:49+00:00
272 982 491/136
1 Toy Story (1995) 5
2 GoldenEye (1995) 3
3 Four Rooms (1995) 4
4 Get Shorty (1995) 3
5 Copycat (1995) 3
['Animation', "Children's", 'Comedy'] ['Action', 'Adventure', 'Thriller']


## Pitfall

User id 196 and the occupation count 196 are different facts. User 196 is one writer. The occupation name `student` belongs to 196 people. The number on the first line of `u.data` does not say how many students are in the file.

## Summary

The first stored row is user 196 giving *Kolya* 3 stars on 4 December 1997. User 1 rated 272 movies, with mean $491/136$, and gave *Toy Story* 5 stars.
